In [1]:
#TODO make sure this renders in the github repo

# Data Loader

The decoder-only model (Llama 3 8B), is trained on next-token prediction using a causal mask: Given a continuous stream of tokens, we slice a fixed-length window (e.g., 256 tokens).

- **Input** ($X$): Tokens $0$ to $N-1$, e.g., $[x_0, x_1, ..., x_{N-1}]$
- **Target** ($Y$): Tokens $1$ to $N$, e.g., $[x_1, x_2, ..., x_{N}]$
- Timestamp example sequence: "This is the Llama 3 8B"
  - The **causal mask** ensures that the tokenized representation for token $i$ is calculated using only the tokens at positions $\leq i$, this prevents the model from "cheating" by looking at the target tokens during training.
  - **Note:** The timestamp table is only for visualization purposes, in practice, the model processes the entire window in one forward pass!

    | Timestamp | Input | Target |
    | --- | --- | --- |
    | 1 | "This" | "is" |
    | 2 | "This is" | "the" |
    | 3 | "This is the" | "Llama" |
    | 4 | "This is the Llama" | "3" |
    | 5 | "This is the Llama 3" | "8B" |

**Example of what the tokens will look like:**

```text
First 10 tokens of the Input:  
    [1, 269, 72, 224, 44, 81, 71, 72, 83, 262]
First 10 tokens of the Target (shifted right): 
    [269, 72, 224, 44, 81, 71, 72, 83, 262, 71]
```

- The dataloader yields dense, packed **batches of tokens**.

## Pipeline

Streaming the dataset from HuggingFace and tokenizing it on the fly is an anti-pattern in large-scale LLM training, it introduces a lot of bottlenecks. The standard pipeline is to:
1. **Offline Pre-processing:** Download the dataset, tokenize the whole dataset, and pack it into fixed length chucks separated by `doc_end_token` prior to training.
2. **Binary Storage:** Save the tokenized outputs into a massive flat 1D array (of uint16, uint32, or higher) in a binary file.
- The `Dataloader` is simply a memory map to the binary file.
- With this method, the size of the dataset stored in memory is minimal. Now most of memory is reserved for storing the model's weights and the massive gradient tensors generated during the backward pass.

⭐️ See [prepare_pretrained.py](../data_prep/prepare_pretrained.py) for the code.

In [2]:
# @i-c
%load_ext autoreload
%autoreload 2

In [3]:
import easyjupyter
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [4]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig

In [5]:
from datasets import load_dataset

#TODO this function is not needed, everywhere its called, just add load_dataset()
def get_raw_ds(name, ds_config):
    """Starts a dataset stream that retrieves raw un-tokenized text from HuggingFace datasets."""
    return load_dataset(name, ds_config, split="train", streaming=True)

/Users/tonyavis/miniconda3/envs/build_an_llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#TODO rename to TokenizedDataset
class PreTokenizeDataset(Dataset):
    def __init__(self, cfg:BaseConfig, bin_path, start_step = 0):
        """
        Retrieve data from a pre-tokenized dataset that is stored in a binary file.

        Args:
            bin_path: Path to the binary file containing the pre-tokenized dataset.
            start_step: The current training step that a checkpoint is at. This is used so that if we resume training a checkpoint, it will start at the point of the dataset that the checkpoint last saw.
        """
        self.cfg = cfg
        # Memory-map the binary file
        self.data = np.memmap(bin_path, dtype=cfg.dtype, mode="r")

        # calculate how many samples (context windows) the model has seen.
        self.samples_seen = start_step *(cfg.micro_batch_size * cfg.gradient_accumulation_steps)

        # Calculate total
        total_possible_samples = (len(self.data) - 1) // cfg.context_len

        # Calculate how many full context windows we have in the dataset.
        self.num_samples = total_possible_samples - self.samples_seen

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        """Get a single context window from the dataset."""
        # Shift the index if needed
        actual_idx = idx + self.samples_seen

        start = actual_idx * self.cfg.context_len
        end = start + self.cfg.context_len + 1

        chunk = torch.from_numpy(self.data[start:end].astype(np.int64))

        # Create the casual mask
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

In [7]:
def create_pretrain_dataloader(cfg, bin_path, start_step=0):
    """
    Args:
        bin_path: The path to the binary file that contains the dataset.
        start_step: The current training step that a checkpoint is at. This is used so that if we resume training a checkpoint, it will start at the point of the dataset that the checkpoint last saw.
    """

    dataset = PreTokenizeDataset(cfg, bin_path, start_step=start_step)

    return DataLoader(
        dataset,
        batch_size=cfg.micro_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True if cfg.device != "mps" else False,
        # pin_memory=True is not supported on mps
    )

## Test

`Run this to create the binary dataset`

```bash
python prepare_data.py --d pretrain --overfit
```

In [12]:
# @i-c
from llama_configs import Llama3_scaled_down
from model.bpe_tokenizer import BPETokenizer

cfg = Llama3_scaled_down()
cfg.max_docs = cfg.overfit_max_docs
cfg.vocab_size = 32_000
cfg.token_budget = 250 * cfg.global_batch_size_tokens

# Get a trained tokenizer
t = BPETokenizer(cfg)
success, tokenizer = t.load_tokenizer()
if not success:
    raise ValueError(
        f"Tokenizer file not found!\n"
        "\n\nPlease run `python prepare.py --d pretrain --overfit` first."
    )


Project Root: /Users/tonyavis/Main/Build_an_LLM

Loading existing trained BPE Tokenizer with vocab size: 32000 (Trained on 1000 documents) from: (/Users/tonyavis/Main/Build_an_LLM/artifacts/universal_tokenizer/bpe_vocab_size_32000_samples_1000.json)...


In [13]:
# @i-c
dataloader = create_pretrain_dataloader(cfg=cfg, bin_path=cfg.DATA_DIR / "overfit_initial_and_lc_stage.bin", start_step=0)

# Fetch a single batch
print("\n\nFetching a single batch from FineWeb-Edu...")
data_iterator = iter(dataloader)
x, y = next(data_iterator)



Fetching a single batch from FineWeb-Edu...


/Users/tonyavis/miniconda3/envs/build_an_llm/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


In [14]:
# @i-c
x.shape, y.shape

(torch.Size([8, 256]), torch.Size([8, 256]))

In [15]:
# @i-c
print("\n\nA single batch of data:")
print(f"x: {x}")
print(f"\ny: {y}")



A single batch of data:
x: tensor([[    1,   449, 16477,  ...,   934, 11538,    17],
        [ 1370,   355,  3282,  ...,  4869,   286,  5172],
        [   17,  7283,    81,  ...,   502,   290, 10990],
        ...,
        [ 1459,   330,   265,  ..., 17880,   290,  1469],
        [   17,  3119,  6034,  ...,   286,   470,  1660],
        [   15,   318,   261,  ...,   334,  3196,   426]])

y: tensor([[  449, 16477,  8444,  ..., 11538,    17,  1370],
        [  355,  3282,   286,  ...,   286,  5172,    17],
        [ 7283,    81,   459,  ...,   290, 10990,   281],
        ...,
        [  330,   265,  1408,  ...,   290,  1469,    17],
        [ 3119,  6034,  1472,  ...,   470,  1660,    15],
        [  318,   261,  3322,  ...,  3196,   426,  1746]])


In [16]:
# @i-c:
print("\n\nDecoding the first 50 tokens of x[0] to view the document:")
token_ids_list = x[0][:50].tolist()
decoded_document = tokenizer.decode(token_ids_list)
print(f"decoded_document: [{decoded_document}...]")



Decoding the first 50 tokens of x[0] to view the document:
decoded_document: [The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins...]


In [17]:
# @i-c:
# Verify the shapes match the llama cfg
print(f"\n\nInput 'x' shape: {x.shape} -> Expected shape: (Micro Batch size, Context length)")
print(f"Input 'y' shape: {y.shape} -> Expected shape: (Micro Batch size, Context length)")



Input 'x' shape: torch.Size([8, 256]) -> Expected shape: (Micro Batch size, Context length)
Input 'y' shape: torch.Size([8, 256]) -> Expected shape: (Micro Batch size, Context length)


In [18]:
# @i-c:
# Verify the target shape are shifted correctly
print(f"\n\nFirst 10 tokens of x[0]: {x[0][:10].tolist()}")
print(f"\nFirst 10 tokens of y[0]: {y[0][:10].tolist()}")



First 10 tokens of x[0]: [1, 449, 16477, 8444, 202, 2056, 502, 265, 3667, 15]

First 10 tokens of y[0]: [449, 16477, 8444, 202, 2056, 502, 265, 3667, 15, 10351]


In [19]:
# @i-c:
print(f"\n\nLast 10 tokens of x[0]: {x[0][-10:].tolist()}")
print(f"\nLast 10 tokens of y[0]: {y[0][-10:].tolist()}")



Last 10 tokens of x[0]: [712, 433, 399, 502, 10665, 281, 1021, 934, 11538, 17]

Last 10 tokens of y[0]: [433, 399, 502, 10665, 281, 1021, 934, 11538, 17, 1370]
